In [1]:
# NOTEBOOK 03: WEIGHTED RISK SCORING & PRODUCT RISK CALCULATION
 
import os
import re

import numpy as np
import pandas as pd
 
 
# ----------------------------------------------------------
# STEP 0: SETUP
# ----------------------------------------------------------
 
PROCESSED_PATH = '../data/processed'
QUALITY_PATH   = '../data/quality_check'
OUTPUT_PATH    = '../data/risk_scoring'
os.makedirs(OUTPUT_PATH, exist_ok=True)
 
PRODUCTS_FILE       = os.path.join(PROCESSED_PATH, 'products_cleaned.csv')
CANONICAL_FILE      = os.path.join(QUALITY_PATH,   'canonical_mapping.csv')
CLASSIFICATION_FILE = os.path.join(QUALITY_PATH,   'classification_top400_COMPLETE.xlsx')
 
print("STEP 0: Setup")
print()
print(f"  {'File':<45} {'Status':>10}")
print(f"  {'----':<45} {'------':>10}")
for f in [PRODUCTS_FILE, CANONICAL_FILE, CLASSIFICATION_FILE]:
    status = 'OK' if os.path.exists(f) else 'MISSING'
    print(f"  {os.path.basename(f):<45} {status:>10}")


STEP 0: Setup

  File                                              Status
  ----                                              ------
  products_cleaned.csv                                  OK
  canonical_mapping.csv                                 OK
  classification_top400_COMPLETE.xlsx                   OK


In [2]:
# ----------------------------------------------------------
# STEP 1: LOAD AND INSPECT INPUTS
# ----------------------------------------------------------
 
print()
print("STEP 1: Load and Inspect Inputs")
print()
 
products       = pd.read_csv(PRODUCTS_FILE)
canonical      = pd.read_csv(CANONICAL_FILE)
classification = pd.read_excel(CLASSIFICATION_FILE)
 
print(f"  {'products_cleaned':<30} {len(products):>8,} rows")
print(f"  {'canonical_mapping':<30} {len(canonical):>8,} rows")
print(f"  {'classification':<30} {len(classification):>8,} rows")
print(f"  {'classified ingredients':<30} "
      f"{classification['risk_level'].notna().sum():>8,}")



STEP 1: Load and Inspect Inputs

  products_cleaned                  7,544 rows
  canonical_mapping                 1,991 rows
  classification                    1,652 rows
  classified ingredients              400


In [3]:
# ----------------------------------------------------------
# STEP 2: CLEAN CLASSIFICATION FILE (safety net)
# ----------------------------------------------------------
# Even though the validator should have caught everything,
# we apply defensive cleaning here so the notebook is
# robust to small data-quality issues.
 
print()
print("STEP 2: Clean Classification (Defensive)")
print()
 
# Strip whitespace from risk_level
classification['risk_level'] = (
    classification['risk_level'].astype('object').apply(
        lambda x: x.strip() if isinstance(x, str) else x
    )
)
 
# Coerce sub_weight to numeric (any stray strings become NaN)
classification['sub_weight'] = pd.to_numeric(
    classification['sub_weight'], errors='coerce'
)
 
print("  Risk level distribution after cleaning:")
for lvl, n in classification['risk_level'].value_counts(dropna=False).items():
    print(f"    {str(lvl):<20} {n:>6,}")



STEP 2: Clean Classification (Defensive)

  Risk level distribution after cleaning:
    nan                   1,252
    Low                     331
    Low-Medium               35
    Medium                   32
    High                      2


In [4]:
# ----------------------------------------------------------
# STEP 3: BUILD CANONICAL -> RISK LOOKUP
# ----------------------------------------------------------
 
print()
print("STEP 3: Build Risk Lookup")
print()
 
classified  = classification[classification['risk_level'].notna()].copy()
risk_lookup = classified.set_index('canonical_name')[
    ['risk_level', 'sub_weight', 'source', 'health_concern']
].to_dict('index')
 
print(f"  {'Classified ingredients in lookup':<40} "
      f"{len(risk_lookup):>8,}")



STEP 3: Build Risk Lookup

  Classified ingredients in lookup              400


In [5]:
# ----------------------------------------------------------
# STEP 4: BUILD RAW -> CANONICAL LOOKUP
# ----------------------------------------------------------
 
print()
print("STEP 4: Build Canonical Mapping Lookup")
print()
 
raw_to_canonical = dict(
    zip(canonical['raw_ingredient'], canonical['canonical_name'])
)
 
print(f"  {'Raw to canonical mappings':<40} "
      f"{len(raw_to_canonical):>8,}")



STEP 4: Build Canonical Mapping Lookup

  Raw to canonical mappings                   1,991


In [6]:
# ----------------------------------------------------------
# STEP 5: SCORING FUNCTION
# ----------------------------------------------------------
# For each product:
#   1. Split ingredients_parsed on '|', clean each token
#   2. Map raw -> canonical via raw_to_canonical
#   3. Look up canonical name in risk_lookup
#   4. Compute weighted average score over classified ingredients only
#   5. Track coverage % and high-risk warnings
 
_PUNCT_RE = re.compile(r'^[\s\*]+|[\s\.\,\;\:]+$')


 
def normalise_token(s):
    """Standardise an ingredient token before lookup."""
    if not isinstance(s, str):
        return ''
    return _PUNCT_RE.sub('', s).strip().lower()
 
 
def score_product(row):
    """Compute the weighted score and per-ingredient detail for one product."""
    raw_text = row['ingredients_parsed']
    if not isinstance(raw_text, str) or not raw_text.strip():
        return None, []
 
    # Split, normalise, deduplicate while preserving order
    seen   = set()
    tokens = []
    for tok in raw_text.split('|'):
        norm = normalise_token(tok)
        if norm and norm not in seen:
            seen.add(norm)
            tokens.append(norm)
 
    long_rows       = []
    weights         = []
    risk_counts     = {'Low': 0, 'Low-Medium': 0, 'Medium': 0,
                       'Medium-High': 0, 'High': 0}
    high_risk_names = []
 
    for raw in tokens:
        canon = raw_to_canonical.get(raw, raw)
        info  = risk_lookup.get(canon)
        if info is None:
            long_rows.append({
                'product_id':     row['product_id'],
                'raw_ingredient': raw,
                'canonical_name': canon if canon != raw else None,
                'risk_level':     'Unclassified',
                'sub_weight':     np.nan,
                'source':         None,
                'health_concern': None,
            })
        else:
            level  = info['risk_level']
            weight = info['sub_weight']
            weights.append(weight)
            if level in risk_counts:
                risk_counts[level] += 1
            if level == 'High':
                high_risk_names.append(canon)
            long_rows.append({
                'product_id':     row['product_id'],
                'raw_ingredient': raw,
                'canonical_name': canon,
                'risk_level':     level,
                'sub_weight':     weight,
                'source':         info['source'],
                'health_concern': info['health_concern'],
            })
 
    total        = len(tokens)
    classified_n = len(weights)
    score        = float(np.mean(weights)) if classified_n > 0 else np.nan
    coverage     = classified_n / total if total > 0 else 0.0
 
    # Risk category from score (per proposal Table 3)
    if pd.isna(score):
        risk_cat = 'Insufficient Data'
    elif score < 1.50:
        risk_cat = 'Low Risk'
    elif score < 2.50:
        risk_cat = 'Medium Risk'
    else:
        risk_cat = 'High Risk'
 
    # Coverage flag
    if coverage >= 0.70:
        cov_flag = 'High'
    elif coverage >= 0.40:
        cov_flag = 'Medium'
    else:
        cov_flag = 'Low'
 
    # Final combined label: never display a confident category
    # when coverage is thin
    if cov_flag == 'Low' or pd.isna(score):
        final_label = 'Insufficient Data'
    elif cov_flag == 'Medium':
        final_label = f'{risk_cat} (Limited Data)'
    else:
        final_label = risk_cat
 
    summary = {
        'product_id':                 row['product_id'],
        'weighted_score':             round(score, 4) if pd.notna(score) else np.nan,
        'risk_category':              risk_cat,
        'coverage_pct':               round(coverage * 100, 2),
        'coverage_flag':              cov_flag,
        'final_risk_label':           final_label,
        'total_ingredients':          total,
        'classified_ingredients':     classified_n,
        'low_count':                  risk_counts['Low'],
        'low_medium_count':           risk_counts['Low-Medium'],
        'medium_count':               risk_counts['Medium'],
        'medium_high_count':          risk_counts['Medium-High'],
        'high_count':                 risk_counts['High'],
        'has_high_risk_warning':      len(high_risk_names) > 0,
        'high_risk_ingredients_list': '; '.join(high_risk_names) if high_risk_names else '',
    }
    return summary, long_rows


In [7]:
# ----------------------------------------------------------
# STEP 6: SCORE EVERY PRODUCT
# ----------------------------------------------------------
 
print()
print("STEP 6: Score Every Product")
print()
 
summaries = []
long_rows = []
 
for _, row in products.iterrows():
    summary, longs = score_product(row)
    if summary is not None:
        summaries.append(summary)
        long_rows.extend(longs)
 
summary_df = pd.DataFrame(summaries)
long_df    = pd.DataFrame(long_rows)
 
print(f"  {'Products scored':<40} {len(summary_df):>8,}")
print(f"  {'Long-format rows built':<40} {len(long_df):>8,}")



STEP 6: Score Every Product

  Products scored                             7,544
  Long-format rows built                    227,213


In [8]:
# ----------------------------------------------------------
# STEP 7: MERGE METADATA INTO SCORE TABLE
# ----------------------------------------------------------
 
print()
print("STEP 7: Merge Product Metadata into Score Table")
print()
 
meta_cols = ['product_id', 'product_name', 'brand_name', 'price_usd',
             'rating', 'reviews', 'loves_count',
             'primary_category', 'secondary_category', 'tertiary_category']
 
product_scores = products[meta_cols].merge(
    summary_df, on='product_id', how='inner'
)
 
# Arrange the final columns in a clear order
ordered = meta_cols + [
    'weighted_score', 'risk_category', 'coverage_pct',
    'coverage_flag', 'final_risk_label',
    'total_ingredients', 'classified_ingredients',
    'low_count', 'low_medium_count', 'medium_count',
    'medium_high_count', 'high_count',
    'has_high_risk_warning', 'high_risk_ingredients_list',
]
product_scores = product_scores[ordered]
 
print(f"  {'Final product score rows':<40} {len(product_scores):>8,}")



STEP 7: Merge Product Metadata into Score Table

  Final product score rows                    7,544


In [9]:
# ----------------------------------------------------------
# STEP 8: BUILD INGREDIENT-LEVEL SUMMARY
# ----------------------------------------------------------
 
print()
print("STEP 8: Build Ingredient-Level Summary")
print()
 
long_classified = long_df[long_df['risk_level'] != 'Unclassified'].copy()
 
# Attach primary category so we can show where each ingredient appears
long_classified = long_classified.merge(
    products[['product_id', 'primary_category']],
    on='product_id', how='left'
)
 
 
def top_categories(series, n=3):
    vc = series.value_counts().head(n)
    return '; '.join(f'{cat} ({cnt})' for cat, cnt in vc.items())
 
 
ingredient_summary = (
    long_classified
        .groupby('canonical_name')
        .agg(
            risk_level=('risk_level', 'first'),
            sub_weight=('sub_weight', 'first'),
            source=('source', 'first'),
            health_concern=('health_concern', 'first'),
            products_containing=('product_id', 'nunique'),
            top_primary_categories=('primary_category', top_categories),
        )
        .reset_index()
        .sort_values(['sub_weight', 'products_containing'],
                     ascending=[False, False])
        .reset_index(drop=True)
)
 
print(f"  {'Distinct classified ingredients found':<40} "
      f"{len(ingredient_summary):>8,}")



STEP 8: Build Ingredient-Level Summary

  Distinct classified ingredients found         400


In [10]:
# ----------------------------------------------------------
# STEP 9: HEADLINE STATISTICS
# ----------------------------------------------------------
 
print()
print("STEP 9: Headline Statistics")
print()
 
print("  --- Weighted score distribution ---")
print(product_scores['weighted_score'].describe().round(3))
print()
print("  --- Final risk label (score adjusted for coverage) ---")
print(product_scores['final_risk_label'].value_counts())
print()
print("  --- Coverage flag distribution ---")
print(product_scores['coverage_flag'].value_counts())
print()
n_warn = product_scores['has_high_risk_warning'].sum()
print(f"  Products with high-risk warning: {n_warn:,} of "
      f"{len(product_scores):,} ({n_warn/len(product_scores)*100:.1f}%)")
print()
print("  --- Average score by primary category ---")
print(
    product_scores
        .dropna(subset=['weighted_score'])
        .groupby('primary_category')
        .agg(n_products=('product_id', 'count'),
             avg_score=('weighted_score', 'mean'),
             avg_coverage=('coverage_pct', 'mean'),
             pct_with_high_risk=('has_high_risk_warning', 'mean'))
        .assign(avg_score=lambda d: d['avg_score'].round(3),
                avg_coverage=lambda d: d['avg_coverage'].round(1),
                pct_with_high_risk=lambda d: (d['pct_with_high_risk'] * 100).round(1))
        .sort_values('avg_score', ascending=False)
)



STEP 9: Headline Statistics

  --- Weighted score distribution ---
count    7367.000
mean        1.217
std         0.234
min         1.000
25%         1.046
50%         1.125
75%         1.294
max         2.000
Name: weighted_score, dtype: float64

  --- Final risk label (score adjusted for coverage) ---
final_risk_label
Low Risk                      3576
Low Risk (Limited Data)       2316
Medium Risk                   1057
Insufficient Data              501
Medium Risk (Limited Data)      94
Name: count, dtype: int64

  --- Coverage flag distribution ---
coverage_flag
High      4633
Medium    2410
Low        501
Name: count, dtype: int64

  Products with high-risk warning: 412 of 7,544 (5.5%)

  --- Average score by primary category ---
                  n_products  avg_score  avg_coverage  pct_with_high_risk
primary_category                                                         
Fragrance               1195      1.657          82.8                12.7
Tools & Brushes            1 

In [11]:
# ----------------------------------------------------------
# STEP 10: SAVE OUTPUTS
# ----------------------------------------------------------
 
print()
print("STEP 10: Save Outputs")
print()
 
out_scores     = os.path.join(OUTPUT_PATH, 'product_risk_scores.csv')
out_long       = os.path.join(OUTPUT_PATH, 'product_ingredient_long.csv')
out_ingredient = os.path.join(OUTPUT_PATH, 'ingredient_risk_summary.csv')
 
product_scores.to_csv(out_scores, index=False)
long_df.to_csv(out_long, index=False)
ingredient_summary.to_csv(out_ingredient, index=False)
 
print(f"  Files saved to: {OUTPUT_PATH}")
print()
for file_path in [out_scores, out_long, out_ingredient]:
    size_mb = os.path.getsize(file_path) / (1024 * 1024)
    print(
        f"    - {os.path.basename(file_path):<35} "
        f"{size_mb:>7.2f} MB"
    )
 
print()



STEP 10: Save Outputs

  Files saved to: ../data/risk_scoring

    - product_risk_scores.csv                1.30 MB
    - product_ingredient_long.csv           19.09 MB
    - ingredient_risk_summary.csv            0.05 MB

